# Customer Journey Graph + Trust Trajectory Score

**Headline insight (plan §6 #3, deck slides 5–6).**

For each customer, every interaction across all three call types stitched chronologically. Trust Trajectory Score is the weighted blend of four components; **always shown with the components and evidence** so the consumer can judge it (plan §1 Failure Mode #3 — never lead with the number alone).

In [ ]:
from ti.insights.customer_journey import rank_customers_by_risk, build_customer_journey
from ti.store import duckdb_store
import pandas as pd

## Top 10 customers by trajectory score (lowest = most at risk)

In [ ]:
top = rank_customers_by_risk(top_n=10)
pd.DataFrame([
    {
        'customer': j.customer_name,
        'score': j.trajectory_score,
        'meetings': len(j.nodes),
        'sentiment_slope': round(j.components.sentiment_slope, 2),
        'closure_rate': j.components.action_closure_rate,
        'escalations': j.components.escalation_count,
        'signal_density': round(j.components.churn_signal_density, 2),
    }
    for j in top
])

## Drill-down: the most at-risk customer

Cross-call timeline + evidence (with utterance text resolved from the DB).

In [ ]:
headline = top[0]
print(f'Customer: {headline.customer_name}')
print(f'Trust Trajectory Score: {headline.trajectory_score:.2f}')
print(f'  sentiment_slope     : {headline.components.sentiment_slope:+.2f}')
print(f'  action_closure_rate : {headline.components.action_closure_rate:.0%}')
print(f'  escalation_count    : {headline.components.escalation_count}')
print(f'  churn_signal_density: {headline.components.churn_signal_density:.2f}')
print()
print('Journey:')
for n in headline.nodes:
    print(f'  {n.date.strftime("%Y-%m-%d")}  [{n.call_type:8s}]  score={n.sentiment_score:.1f}  risks={n.risk_signal_count}  — {n.title}')

In [ ]:
print('Evidence (citations):')
with duckdb_store.connect() as con:
    for c in headline.evidence:
        row = con.execute(
            'SELECT speaker_name, text FROM utterances WHERE meeting_id=? AND utt_index=?',
            [c.meeting_id, c.utterance_index],
        ).fetchone()
        if row:
            print(f'  · {c.meeting_id[-8:]} utt#{c.utterance_index} — {row[0]}: {row[1][:140]}')